In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import joblib
from sklearn.preprocessing import StandardScaler

normal_conn_paths = sorted(glob.glob("../data/zeek_logs/normal_*/conn.log"))

print("Liczba plików normal conn.log:", len(normal_conn_paths))
print("Pierwsze pliki:")
for p in normal_conn_paths[:10]:
    print(" ", p)

cols = [
    "duration",
    "orig_bytes",
    "resp_bytes",
    "proto",
    "conn_state",
    "orig_pkts",
    "resp_pkts",
]

numeric_cols = ["duration", "orig_bytes", "resp_bytes", "orig_pkts", "resp_pkts"]

def read_zeek_conn(path):
    with open(path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    header = None
    data_lines = []

    for line in lines:
        line = line.strip()

        if not line:
            continue

        if line.startswith("#fields"):
            header = line.split("\t")[1:]
            continue

        if line.startswith("#"):
            continue

        data_lines.append(line.split("\t"))

    if header is None:
        raise ValueError(f"Nie znaleziono #fields w pliku: {path}")

    return pd.DataFrame(data_lines, columns=header)

all_dfs = []

for path in normal_conn_paths:
    df = read_zeek_conn(path)

    missing = [c for c in cols if c not in df.columns]
    if missing:
        print("Pomijam, brak kolumn:", path, missing)
        continue

    df = df[cols].copy()
    df["source"] = os.path.basename(os.path.dirname(path))
    all_dfs.append(df)

    print(f"{path} -> flowy: {len(df)}")

normal_raw = pd.concat(all_dfs, ignore_index=True)

print("\n=== PODSUMOWANIE NORMAL ===")
print("Łącznie flowów normalnych:", len(normal_raw))
print("Liczba źródeł:", normal_raw["source"].nunique())
print("\nTop 20 źródeł:")
print(normal_raw["source"].value_counts().head(20))

os.makedirs("../data/processed", exist_ok=True)

normal_raw.to_csv("../data/processed/normal_raw_combined.csv", index=False)

df2 = normal_raw.drop(columns=["source"]).copy()

for col in numeric_cols:
    df2[col] = pd.to_numeric(df2[col], errors="coerce").fillna(0)

for col in ["duration", "orig_bytes", "resp_bytes"]:
    df2[col] = np.log1p(df2[col])

df2 = pd.get_dummies(df2, columns=["proto", "conn_state"], drop_first=True)

feature_columns = df2.columns.tolist()

pd.DataFrame({"feature": feature_columns}).to_csv(
    "../data/processed/feature_schema.csv",
    index=False
)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df2)

joblib.dump(scaler, "../data/processed/standard_scaler.pkl")

normal_features = pd.DataFrame(X_scaled, columns=feature_columns)
normal_features.to_csv("../data/processed/normal_features.csv", index=False)

print("\n=== ZAPISANO ===")
print("../data/processed/normal_raw_combined.csv")
print("../data/processed/normal_features.csv")
print("../data/processed/feature_schema.csv")
print("../data/processed/standard_scaler.pkl")
print("\nShape normal_features:", normal_features.shape)
print("Liczba cech:", len(feature_columns))